# 地理表认识与邮编覆盖

本课先检查地理原表中的邮编前缀、城市、州和坐标，再估算客户表与卖家表按邮编前缀关联时会产生多少行。

| 来源表 | 一行的含义 | 本课用到的字段 |
| --- | --- | --- |
| 地理表 | 一条邮编前缀、城市、州和坐标的来源记录 | 五个原始字段 |
| 客户表 | 一个客户记录 | customer_zip_code_prefix |
| 卖家表 | 一个卖家记录 | seller_zip_code_prefix |

把邮编前缀当作字符串读取，可以保留开头的零。


## 1. 读入三张原表

地理原表保留坐标文本，下一节再另行转换。客户和卖家各读入一个邮编前缀列，节省内存。


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从项目根目录或 notebooks 目录打开本课')

data_dir = project_dir.parent / 'archive'
read_options = {
    'dtype': str, 'encoding': 'utf-8-sig',
    'keep_default_na': False, 'na_values': [''],
}
zip_col = 'geolocation_zip_code_prefix'
lat_col = 'geolocation_lat'
lng_col = 'geolocation_lng'
state_col = 'geolocation_state'

geo_raw = pd.read_csv(data_dir / 'olist_geolocation_dataset.csv', **read_options)
customer_zip = pd.read_csv(
    data_dir / 'olist_customers_dataset.csv',
    usecols=['customer_zip_code_prefix'], **read_options
)['customer_zip_code_prefix']
seller_zip = pd.read_csv(
    data_dir / 'olist_sellers_dataset.csv',
    usecols=['seller_zip_code_prefix'], **read_options
)['seller_zip_code_prefix']

display(geo_raw.head())
print('地理原表行列数', geo_raw.shape)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1,01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
2,01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
3,01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
4,01035,-23.541577961711493,-46.64160722329613,sao paulo,SP


地理原表行列数 (1000163, 5)


## 2. 区分整行重复与邮编前缀重复

duplicated() 对整行比较，统计每组相同记录从第二条开始的行数。
value_counts() 按邮编前缀计数。一个前缀下可以有多条地理记录，其中的坐标可能不同。

先计算整行去重后的邮编前缀分布，看看同一个前缀是否仍对应多行。


In [2]:
geo_zip = geo_raw[zip_col]
geo_rows_per_prefix = geo_zip.value_counts()
duplicate_rows = geo_raw.duplicated()

# 用去重后的行做对照计数。坐标仍使用来源字段。
distinct_row_counts = geo_raw.loc[~duplicate_rows, zip_col].value_counts()
coordinate_counts = (
    geo_raw[[zip_col, lat_col, lng_col]]
    .drop_duplicates()
    .groupby(zip_col).size()
)
state_counts = geo_raw.groupby(zip_col)[state_col].nunique()

display(pd.Series({
    '原表行数': len(geo_raw),
    '不同邮编前缀': geo_zip.nunique(),
    '邮编前缀缺失行数': geo_zip.isna().sum(),
    '邮编前缀格式异常行数': (geo_zip.notna() & ~geo_zip.str.fullmatch('[0-9]{5}', na=False)).sum(),
    '邮编前缀以0开头的行数': geo_zip.str.startswith('0', na=False).sum(),
    '整行重复数（首条之后）': duplicate_rows.sum(),
    '整行去重后的行数': len(geo_raw) - duplicate_rows.sum(),
    '整行去重后仍对应多行的邮编前缀数': distinct_row_counts.gt(1).sum(),
    '对应多对不同坐标值的邮编前缀数': coordinate_counts.gt(1).sum(),
    '对应多个州标签的邮编前缀数': state_counts.gt(1).sum(),
}))

# 取记录数最多的前缀，查看该前缀下几条不同的来源记录。
example_zip = geo_rows_per_prefix.idxmax()
print('样本邮编前缀', example_zip)
print('该前缀在原表的记录数', geo_rows_per_prefix.loc[example_zip])
display(geo_raw.loc[geo_zip.eq(example_zip)].drop_duplicates().head())


原表行数                1000163
不同邮编前缀                19015
邮编前缀缺失行数                  0
邮编前缀格式异常行数                0
邮编前缀以0开头的行数          245733
整行重复数（首条之后）          261831
整行去重后的行数             738332
整行去重后仍对应多行的邮编前缀数      17823
对应多对不同坐标值的邮编前缀数       17781
对应多个州标签的邮编前缀数             8
dtype: int64

样本邮编前缀 24220
该前缀在原表的记录数 1146


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
470805,24220,-22.905816548704518,-43.106988856771984,niteroi,RJ
470811,24220,-22.902305918691127,-43.11254462280002,niteroi,RJ
470812,24220,-22.904566594523498,-43.11049125088019,niteroi,RJ
470820,24220,-22.902574860407324,-43.109192038980986,niteroi,RJ
470821,24220,-22.90749962932978,-43.10616955390452,niteroi,RJ


## 3. 转换坐标并检查取值

errors='coerce' 会把无法转换的文本变成缺失值。结合转换前的文本，分别检查原始缺失和解析失败。
本格检查全球经纬度取值范围，纬度为 −90 到 90，经度为 −180 到 180。坐标是否与记录中的巴西地区一致，留到代表坐标整理时核查。


In [3]:
latitude = pd.to_numeric(geo_raw[lat_col], errors='coerce')
longitude = pd.to_numeric(geo_raw[lng_col], errors='coerce')

parse_failed = (
    (geo_raw[lat_col].notna() & latitude.isna())
    | (geo_raw[lng_col].notna() & longitude.isna())
)
both_parsed = latitude.notna() & longitude.notna()
outside_world = both_parsed & (
    ~latitude.between(-90, 90) | ~longitude.between(-180, 180)
)

display(geo_raw.isna().sum().rename('原始缺失数'))
display(pd.Series({
    '任一坐标非空但解析失败的行数': parse_failed.sum(),
    '超出全球经纬度范围的行数': outside_world.sum(),
}))
display(pd.DataFrame({
    '最小值': [latitude.min(), longitude.min()],
    '最大值': [latitude.max(), longitude.max()],
}, index=['纬度', '经度']))


geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
Name: 原始缺失数, dtype: int64

任一坐标非空但解析失败的行数    0
超出全球经纬度范围的行数      0
dtype: int64

,最小值,最大值
纬度,-36.605374,45.065933
经度,-101.466766,121.105394


## 4. 检查邮编覆盖与左关联行数

map 会为客户或卖家的每条记录查找该前缀在地理原表中的行数。匹配 3 行的记录在左关联后贡献 3 行，匹配不到的记录在左关联后保留 1 行。
据此可以估算左关联总行数。覆盖统计以邮编前缀为键，匹配成功表示前缀存在于地理原表。


In [4]:
def check_zip_coverage(prefixes):
    # 每条左表记录对应的右表匹配数。找不到前缀时为缺失。
    matching_rows = prefixes.map(geo_rows_per_prefix)
    unmatched = matching_rows.isna()

    # 左关联保留未匹配的行，因此把这些记录的输出行数按 1 计算。
    estimated_rows = int(matching_rows.fillna(1).sum())
    return {
        '原表行数': len(prefixes),
        '不同邮编前缀': prefixes.nunique(),
        '未匹配记录数': unmatched.sum(),
        '未匹配的不同邮编前缀数': prefixes.loc[unmatched].nunique(),
        '匹配地理表多行的记录数': matching_rows.gt(1).sum(),
        '直接左关联预计行数': estimated_rows,
    }

zip_coverage = pd.DataFrame({
    '客户表': check_zip_coverage(customer_zip),
    '卖家表': check_zip_coverage(seller_zip),
})
display(zip_coverage)


,客户表,卖家表
原表行数,99441,3095
不同邮编前缀,14994,2246
未匹配记录数,278,7
未匹配的不同邮编前缀数,157,7
匹配地理表多行的记录数,98976,3079
直接左关联预计行数,15083733,435094


## 我的理解

逐格运行，发来最后一格的邮编覆盖表，并回答三道题。

1. 地理表有多少行、多少个不同的邮编前缀？去掉整行重复后，邮编前缀能作为主键吗？
地理表有1000163行、19015个不同的邮编前缀；不能作为主键，因为整行去重后仍对应多行的邮编前缀数有17823，所以不唯一。
2. 一条客户记录匹配地理表 3 行，另一条客户记录找不到匹配。这两条记录做左关联一共会得到几行？
4行
3. 客户表的「未匹配记录数」和「未匹配的不同邮编前缀数」分别数什么？为什么两者可能不同？
未匹配记录数278条，未匹配的不同邮编前缀数157条；未匹配记录数表示客户表没有匹配地理表的行，同一个未匹配邮编前缀可以出现在多条客户记录中，所以 278 条未匹配记录按前缀去重后得到 157 个不同前缀。

下一步会检查坐标异常和邮编地区冲突，再讨论每组如何选取代表坐标。
